In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Unzip the dataset (Change 'prescription_dataset.zip' to your actual file name)
!unzip -q "/content/drive/MyDrive/prescription.zip" -d "/content/dataset/"

In [7]:
import tensorflow as tf
import os

# Check if GPU is active
print("GPU Available:", tf.config.list_physical_devices('GPU'))

# Check if folders unzipped correctly
print("Folders found:", os.listdir("/content/dataset/"))

GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Folders found: ['data', '__huggingface_repos__.json']


In [18]:
import pandas as pd
import os
import json

# 1- defining the basepath where dataset was unzipped
dataset_dir = "/content/dataset/data/processed/"

# 2- load the json files that maps numbers to drug names
json_path = os.path.join(dataset_dir,'mapping.json')

with open(json_path, 'r') as f:
    drug_map = json.load(f)

# printing out the first 5 items to see how it looks
print("---sample of json mapping----")
sample_keys = list(drug_map.keys())[:5]

for key in sample_keys:
  print(f"Number {drug_map[key]} maps to drug: {key}")

print(f"Total Number of Drug Classes in JSON: {len(drug_map)}")


---sample of json mapping----
Number 0 maps to drug: Aceta
Number 1 maps to drug: Ace
Number 2 maps to drug: Alatrol
Number 3 maps to drug: Amodis
Number 4 maps to drug: Atrizin
Total Number of Drug Classes in JSON: 78


In [16]:
# loading the train, test, val csv files

train_csv_path = os.path.join(dataset_dir, 'train_labels.csv')
test_csv_path = os.path.join(dataset_dir,'test_labels.csv')
val_csv_path = os.path.join(dataset_dir,'val_labels.csv')

train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)
val_df = pd.read_csv(val_csv_path)

# inspecting the top five rows of the training dataframe
print(train_df.head())

print("\n--Dataset sizes---")
print(f"Training Set Size: {len(train_df)}")
print(f"Test Set Size: {len(test_df)}")
print(f"Validation Set Size: {len(val_df)}")

   IMAGE  MEDICINE_NAME
0  0.png              0
1  1.png              0
2  2.png              0
3  3.png              0
4  4.png              0

--Dataset sizes---
Training Set Size: 3120
Test Set Size: 780
Validation Set Size: 780


In [19]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_df['MEDICINE_NAME'] = train_df['MEDICINE_NAME'].astype(str)
val_df['MEDICINE_NAME'] = val_df['MEDICINE_NAME'].astype(str)
test_df['MEDICINE_NAME'] = test_df['MEDICINE_NAME'].astype(str)

# Defining the ImageDataGenerator
# this step will automatically rescale the pixel values from [0, 255] to [0.0, 1.0]

train_datagen = ImageDataGenerator(rescale=1./255)
val_test_datagen = ImageDataGenerator(rescale=1./255)

# creating the TrainingDataGenerator
train_generator = train_datagen.flow_from_dataframe(
    dataframe = train_df,
    directory = "/content/dataset/data/processed/train/images",
    x_col = "IMAGE",
    y_col = "MEDICINE_NAME",
    target_size = (84, 84),
    color_mode ="grayscale",
    class_mode = "categorical",
    batch_size = 32,
    shuffle = True,
)

# creating validation Data Generator
validation_generator = val_test_datagen.flow_from_dataframe(
    dataframe = val_df,
    directory = "/content/dataset/data/processed/val/images",
    x_col = "IMAGE",
    y_col = "MEDICINE_NAME",
    target_size = (84, 84),
    color_mode ="grayscale",
    class_mode = "categorical",
    batch_size = 32,
    shuffle = False,
)

# creating test Data Generator
test_generator = val_test_datagen.flow_from_dataframe(
    dataframe = test_df,
    directory = "/content/dataset/data/processed/test/images",
    x_col = "IMAGE",
    y_col = "MEDICINE_NAME",
    target_size = (84, 84),
    color_mode ="grayscale",
    class_mode = "categorical",
    batch_size = 32,
    shuffle = False,
)



Found 3120 validated image filenames belonging to 78 classes.
Found 780 validated image filenames belonging to 78 classes.
Found 780 validated image filenames belonging to 78 classes.


In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
import io

# Initializing the Sequential Model Container
model = Sequential()

# Block 1 (detecting low level strokes)
# Input shape is (84, 84, 1), since image size is 84 x 84 and 1 channel (greyscale)
model.add(Conv2D(32, kernel_size=(3,3), activation= 'relu', input_shape=(84,84,1)))
model.add(BatchNormalization()) # this will stablize and speed up the training
model.add(MaxPooling2D(pool_size =(2,2))) # pooling will skrink the size to 41x41
model.add(Dropout(0.25))

# Block 2 (detecting Medium Level shapes)
model.add(Conv2D(64, kernel_size= (3,3), activation ='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2))) # shrink the image size to 19x19
model.add(Dropout(0.25))

# Block 3 (detecting high level shapes)
model.add(Conv2D(128, kernel_size= (3,3), activation ='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2))) # shrink the image size to 8x8
model.add(Dropout(0.25))

# Block 4 (The Classification Brain)
model.add(Flatten()) # Flattens the 8x8 matrix of features into a long 1D vector of numbers

model.add(Dense(256, activation ='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))

# Atlast the output layers must have 78 Nodes because we have 78 unique medicine classes
# using the 'softmax' because it converts the outputs into percentages/probabilites that add upto 1.0 (for example 85% chance that it is Amoxicillin)
model.add(Dense(78, activation ='softmax'))


# Compile the Model
model.compile(
    optimizer = 'adam',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)

stream = io.StringIO()
model.summary(print_fn=lambda x: stream.write(x + '\n'))
summary_string = stream.getvalue()

print(summary_string)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 82, 82, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 82, 82, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 41, 41, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 41, 41, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 39, 39, 64)